In [36]:
from __future__ import annotations

import errno

import lmdb
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from meta import Run, Dir, File

In [110]:
db_name = 'test-db'
path = Path(os.curdir) / db_name

create=True
create=False

readonly = True
readonly = False

try:
    env = lmdb.open(str(path), map_size=2**16, subdir=True, create=create, readonly=readonly, max_readers=3)
except lmdb.Error as exc:
    # print(dir(exc))
    if re.match('.*cannot find the path specified.*', exc.args[0]):
        print(7)
    else:
        raise exc


Error: The environment 'test-db' is already open in this process.

In [135]:
env, env.flags(), env.path(), env.info()

(<Environment at 0x2c44c0b13e0>,
 {'subdir': True,
  'readonly': False,
  'metasync': True,
  'sync': True,
  'map_async': False,
  'readahead': True,
  'writemap': False,
  'meminit': False,
  'lock': True},
 'test-db',
 {'map_addr': 0,
  'map_size': 4294967296,
  'last_pgno': 7,
  'last_txnid': 10,
  'max_readers': 126,
  'num_readers': 1})

In [136]:
env.close()

In [139]:
run_id = '0047'
run = Run(
    run_id,
    description='test write Run rec',
    specification='',
    start_time=1.234,
    end_time=1.234,
    duration=1.234,
    status='pending')

def mk_run_key(run):
    return bytes(f'r:{run_id}', 'utf-8')

def mk_run_rec(run):
    return bytes(run.description, 'utf-8')

key = mk_run_key(run)
value = mk_run_rec(run)

key, value


(b'r:0047', b'test write Run rec')

In [140]:
with env.begin(write=True) as txn:
    txn.put(key, value)

txn

In [134]:
txn
# txn.stat()

In [141]:
with env.begin() as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            print(key, value)

b'0042' b'test write Run rec'
b'0049' b'test write Run rec'
b'r:0042' b'test write Run rec'
b'r:0047' b'test write Run rec'
b'r:0049' b'test write Run rec'
b'r:0050' b'test write Run rec'


In [138]:
from db import Db

path = Path('test-db')
readonly = False

db = Db.open(path, readonly, create=False)
env = db.env